# GLOSSATE - personal run

Takes cleaned videos from my Drive `ELUATE/output` folder, creates translated subtitles, and optionally writes hard-subtitled videos to `GLOSSATE/output`. Both folders sync to my Mac via Google Drive, so the results appear locally without a manual download.

Runtime: `Runtime -> Change runtime type -> GPU`. Run the cells top to bottom.


## 1. Install GLOSSATE


In [ ]:
GLOSSATE_INSTALL = "glossate"

!python -m pip install -q --upgrade pip
!python -m pip install -q --upgrade "{GLOSSATE_INSTALL}[cuda,detect]"

!glossate --version
!ffmpeg -version | head -n 1


## 2. Mount Google Drive


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 3. Configure and list cleaned videos


In [ ]:
from pathlib import Path
import torch

# Input is the cleaned video output from the ELUATE personal notebook.
INPUT_DIR  = "/content/drive/MyDrive/ELUATE/output"
OUTPUT_DIR = "/content/drive/MyDrive/GLOSSATE/output"

SOURCE_LANG = None
TARGET_LANG = "tr"
SUBTITLE_FORMAT = "srt"
BURN_SUBTITLES = True

# CUDA/Colab defaults. For local Ollama/Gemma outside Colab, use:
# MT_MODEL = "ollama/gemma4:e4b" and MT_BACKEND = "ollama".
ASR_MODEL = "turbo"
MT_MODEL = "nllb-200-600m"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ASR_BACKEND = "faster-whisper" if DEVICE == "cuda" else "auto"
MT_BACKEND = "nllb"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

VIDEO_EXTS = {".mp4", ".mov", ".mkv", ".webm", ".m4v", ".avi"}

input_dir = Path(INPUT_DIR)
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

if not input_dir.exists():
    raise FileNotFoundError(
        f"Input folder not found: {input_dir}. Run ELUATE first or check the Drive folder name."
    )

videos = sorted(p for p in input_dir.iterdir() if p.suffix.lower() in VIDEO_EXTS)

print(f"Device: {DEVICE}")
print(f"Input:  {input_dir}")
print(f"Output: {output_dir}")
print(f"Found {len(videos)} cleaned video(s):")
for v in videos:
    print(f"  - {v.name}")
if not videos:
    print("No videos found. Run ELUATE first, then re-run this cell.")


## 4. Check CUDA runtime

The first GLOSSATE run downloads model weights into the Colab runtime cache.


In [ ]:
assert torch.cuda.is_available(), "CUDA is not available. In Colab, choose Runtime -> Change runtime type -> GPU."
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

!nvidia-smi
!glossate info


## 5. Subtitle every cleaned video


In [ ]:
import glossate
from tqdm.auto import tqdm

results = []
target_label = TARGET_LANG or "source"

with glossate.Session(
    asr_model=ASR_MODEL,
    mt_model=MT_MODEL,
    device=DEVICE,
    asr_backend=ASR_BACKEND,
    mt_backend=MT_BACKEND,
    compute_type=COMPUTE_TYPE,
) as session:
    for video in tqdm(videos, desc="all videos", unit="file"):
        subtitle_path = output_dir / f"{video.stem}.{target_label}.{SUBTITLE_FORMAT}"
        burned_path = output_dir / f"{video.stem}.{target_label}.subbed.mp4"

        if BURN_SUBTITLES:
            output = session.subtitle_video(
                video,
                source=SOURCE_LANG,
                target=TARGET_LANG,
                format=SUBTITLE_FORMAT,
                subtitle_output=subtitle_path,
                output=burned_path,
            )
            results.append((video, subtitle_path, output))
        else:
            output = session.subtitle(
                video,
                source=SOURCE_LANG,
                target=TARGET_LANG,
                format=SUBTITLE_FORMAT,
                output=subtitle_path,
            )
            results.append((video, output, None))

print("\nDone.")
for video, subtitle_path, burned_path in results:
    print(f"\n{video.name}")
    print(f"  subtitles: {subtitle_path}")
    if burned_path is not None:
        print(f"  video:     {burned_path}")
print(f"\nOutputs are in {output_dir} and sync back to your Mac via Drive.")
